<a href="https://colab.research.google.com/github/rishitachoudhury7/clinical-trial-matching/blob/main/clinical_trial_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [9]:
!pip install sentence-transformers transformers datasets accelerate
!pip install rank_bm25 pandas pyarrow scikit-learn requests tqdm
!pip install ir_datasets

In [10]:
import sentence_transformers, transformers, datasets, ir_datasets, rank_bm25, pandas, pyarrow
print("all good")

all good


In [12]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("clinical-trial-matching"))   # use the exact secret name here

##

In [13]:
import ir_datasets, pandas as pd

ds = ir_datasets.load("clinicaltrials/2021/trec-ct-2021")

# queries = patient descriptions
queries = {q.query_id: q.text for q in ds.queries_iter()}
print("queries:", len(queries))

# qrels = relevance judgments (0 = not relevant, 1 = excluded, 2 = eligible)
qrels = pd.DataFrame(ds.qrels_iter())          #
print("qrels:", len(qrels))
print(qrels["relevance"].value_counts())

# trials
docs = {}
for d in ds.docs_iter():
    docs[d.doc_id] = d
print("trials:", len(docs))
print(docs[next(iter(docs))]._fields)

[INFO] [starting] http://www.trec-cds.org/topics2021.xml
[INFO] [finished] http://www.trec-cds.org/topics2021.xml: [00:00] [64.6kB] [536kB/s]


queries: 75


[INFO] [starting] https://trec.nist.gov/data/trials/qrels2021.txt
[INFO] [finished] https://trec.nist.gov/data/trials/qrels2021.txt: [00:00] [676kB] [2.66MB/s]
[INFO] [starting] building docstore


qrels: 35832
relevance
0    24243
1     6019
2     5570
Name: count, dtype: int64


[INFO] If you have a local copy of http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/e12eb9a0d21452503b0ef8874c69f490
[INFO] [starting] http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip
docs_iter:   0%|                                    | 0/375580 [00:00<?, ?doc/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.0%| 0.00/383M [00:00<?, ?B/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.0%| 16.4k/383M [00:00<1:02:47, 102kB/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.0%| 49.2k/383M [00:00<42:37, 150kB/s]  
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.0%| 106k/383M [00:00<30:05, 212kB/s] 
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.1%| 205k/383M [00:00<21:03, 303kB/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-2

trials: 375580
('doc_id', 'title', 'condition', 'summary', 'detailed_description', 'eligibility')


In [14]:
qid = next(iter(queries))
print(queries[qid][:600])

d = docs[qrels[qrels.query_id == qid].doc_id.iloc[0]]
print(d.title, "\n", d.condition, "\n", d.eligibility[:800])


Patient is a 45-year-old man with a history of anaplastic astrocytoma of the spine complicated by severe lower extremity weakness and urinary retention s/p Foley catheter, high-dose steroids, hypertension, and chronic pain. The tumor is located in the T-L spine, unresectable anaplastic astrocytoma s/p radiation. Complicated by progressive lower extremity weakness and urinary retention. Patient initially presented with RLE weakness where his right knee gave out with difficulty walking and right anterior thigh numbness. MRI showed a spinal cord conus mass which was biopsied and found to be anap
Radiation Therapy With or Without Chemotherapy in Treating Patients With Anaplastic Oligodendroglioma 
  
 
      
        DISEASE CHARACTERISTICS:

          -  Histologically proven unifocal or multifocal, supratentorial, pure or mixed anaplastic
             oligodendroglioma

               -  Prior suspected or proven low-grade glioma allowed if current histologic proof of
                  

In [16]:
import textwrap

for qid in list(queries)[:3]:                      # first 3 patients
    print("=" * 100)
    print("QUERY", qid, ":", textwrap.shorten(queries[qid], 300))
    sub = qrels[qrels.query_id == qid]
    print("judged trials:", len(sub), "| label counts:", sub.relevance.value_counts().to_dict())

    for rel in [2, 1, 0]:                          # eligible, excluded, not relevant
        row = sub[sub.relevance == rel].head(1)
        if len(row):
            d = docs[row.doc_id.iloc[0]]
            print(f"  [label {rel}] {d.doc_id}: {d.title}")

QUERY 1 : Patient is a 45-year-old man with a history of anaplastic astrocytoma of the spine complicated by severe lower extremity weakness and urinary retention s/p Foley catheter, high-dose steroids, hypertension, and chronic pain. The tumor is located in the T-L spine, unresectable anaplastic [...]
judged trials: 407 | label counts: {0: 238, 1: 122, 2: 47}
  [label 2] NCT00002814: Combination Chemotherapy for Patients With Brain Cancer
  [label 1] NCT00002569: Radiation Therapy With or Without Chemotherapy in Treating Patients With Anaplastic Oligodendroglioma
  [label 0] NCT00002806: Chemotherapy Followed by Radiation Therapy in Treating Adults With Supratentorial Glioma
QUERY 2 : 48 M with a h/o HTN hyperlipidemia, bicuspid aortic valve, and tobacco abuse who presented to his cardiologist on [**2148-10-1**] with progressive SOB and LE edema. TTE revealed severe aortic stenosis with worsening LV function. EF was 25%. RV pressure was 41 and had biatrial enlargement. [...]
judged tri